In [1]:
from transformers import AutoTokenizer, LlamaForCausalLM

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

/Users/alibayram/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from llama3_2.model import Llama3Model, LLAMA32_CONFIG_1B

llama_fs_model = Llama3Model(LLAMA32_CONFIG_1B)

In [3]:
import torch
import torch.nn as nn

fn = nn.SiLU()
fn2 = nn.functional.silu

In [4]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
llama_model = llama_model.to(device)

In [6]:
llama_fs_model = llama_fs_model.to(device)

In [7]:
example_input_fn = torch.randn(1, 4, 8)
example_output_fn = fn(example_input_fn)
print(example_output_fn)
example_output_fn = fn2(example_input_fn)
print(example_output_fn)

tensor([[[ 0.2917,  1.3253,  0.3829, -0.0156,  0.1266, -0.2378, -0.1990,
          -0.1948],
         [-0.2767,  0.6588, -0.1866, -0.2768, -0.1832, -0.2752, -0.0627,
          -0.2732],
         [-0.2514, -0.1789,  0.2275,  0.4072, -0.0897,  0.8898,  0.6131,
          -0.2197],
         [-0.2307, -0.1102,  2.3252,  0.6749, -0.1073, -0.2074,  0.0939,
          -0.1578]]])
tensor([[[ 0.2917,  1.3253,  0.3829, -0.0156,  0.1266, -0.2378, -0.1990,
          -0.1948],
         [-0.2767,  0.6588, -0.1866, -0.2768, -0.1832, -0.2752, -0.0627,
          -0.2732],
         [-0.2514, -0.1789,  0.2275,  0.4072, -0.0897,  0.8898,  0.6131,
          -0.2197],
         [-0.2307, -0.1102,  2.3252,  0.6749, -0.1073, -0.2074,  0.0939,
          -0.1578]]])


In [8]:
token = llama_tokenizer.encode("Hello")
token = torch.tensor([[9906]]).to(device)
token

tensor([[9906]], device='mps:0')

In [9]:
embedding = llama_model.model.embed_tokens(token)
embedding.shape

torch.Size([1, 1, 2048])

In [10]:
model_output = llama_model.model(token)
model_output

BaseModelOutputWithPast(last_hidden_state=tensor([[[-0.0714, -0.0281,  1.7906,  ...,  0.5309, -0.6187,  0.3760]]],
       device='mps:0', grad_fn=<MulBackward0>), past_key_values=<transformers.cache_utils.DynamicCache object at 0x16bc8b770>, hidden_states=None, attentions=None)

In [11]:
llama_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [12]:
from llama3_2.config import LlamaConfig
config = LlamaConfig()


In [13]:
from llama3_2.model_trfmrs import LlamaForCausalLM
llama_trfmrs_csl_model = LlamaForCausalLM(config)
llama_trfmrs_csl_model.load_state_dict(llama_model.state_dict())
llama_trfmrs_csl_model = llama_trfmrs_csl_model.to(device)
llama_trfmrs_csl_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=Fal

In [14]:
from llama3_2.model_trfmrs import LlamaModel


llama_trfmrs_model = LlamaModel(config)
llama_trfmrs_model = llama_trfmrs_model.to(device)
llama_trfmrs_model

LlamaModel(
  (embed_tokens): Embedding(128256, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [15]:
llama_trfmrs_model.load_state_dict(llama_model.model.state_dict())

<All keys matched successfully>

In [16]:
llama_trfmrs_model_output = llama_trfmrs_model(token)
llama_trfmrs_model_output.shape

torch.Size([1, 1, 2048])

In [17]:
model_output.last_hidden_state

tensor([[[-0.0714, -0.0281,  1.7906,  ...,  0.5309, -0.6187,  0.3760]]],
       device='mps:0', grad_fn=<MulBackward0>)

In [18]:
llama_trfmrs_model_output

tensor([[[-0.0744, -0.0225,  1.7818,  ...,  0.5319, -0.6204,  0.3711]]],
       device='mps:0', grad_fn=<MulBackward0>)

In [19]:
def count_parameters(model):
    """Count the number of parameters in a model"""
    total_params = sum(p.numel() for p in model.parameters())
    return total_params

In [20]:
# Count parameters for the llama_model
total_params = count_parameters(llama_model.model)
print(f"Total parameters in llama_model: {total_params:,}")

Total parameters in llama_model: 1,235,814,400


In [21]:
total_params = count_parameters(llama_trfmrs_model)
print(f"Total parameters in llama_trfmrs_model: {total_params:,}")

Total parameters in llama_trfmrs_model: 1,235,814,400


In [22]:
total_params = count_parameters(llama_trfmrs_csl_model)
print(f"Total parameters in llama_trfmrs_csl_model: {total_params:,}")

Total parameters in llama_trfmrs_csl_model: 1,498,482,688


In [23]:
llama_trfmrs_csl_model_out = llama_trfmrs_csl_model(token)
llama_trfmrs_csl_model_out.shape

torch.Size([1, 1, 128256])

In [24]:
llama_model_out = llama_model(token)
llama_model_out

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 5.6203,  5.5937,  7.6274,  ..., -5.0293, -5.0295, -5.0299]]],
       device='mps:0', grad_fn=<LinearBackward0>), past_key_values=<transformers.cache_utils.DynamicCache object at 0x16bd9d810>, hidden_states=None, attentions=None)

In [25]:
predicted_token = torch.argmax(llama_model_out.logits[0, -1, :], dim=-1)
predicted_token

llama_tokenizer.decode(predicted_token)

','

In [26]:
predicted_token = torch.argmax(llama_trfmrs_csl_model_out[0, -1, :], dim=-1)
predicted_token

llama_tokenizer.decode(predicted_token)

','

In [27]:
llama_trfmrs_csl_model_out

tensor([[[ 5.6023,  5.5664,  7.5911,  ..., -5.0175, -5.0178, -5.0182]]],
       device='mps:0', grad_fn=<LinearBackward0>)